In [123]:
import os
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from xgboost import XGBRegressor


## **Load Data**

In [124]:
print(f"Current working directory: {os.getcwd()}")
os.chdir("/Users/derekwu/Desktop/Data/01_silver")
print(f"Current working directory: {os.getcwd()}")

data = pd.read_csv("clean_chs_waittimes.csv")

Current working directory: /Users/derekwu/Desktop/Data/01_silver
Current working directory: /Users/derekwu/Desktop/Data/01_silver


## **Encoding**

In [125]:
from sklearn.preprocessing import LabelEncoder

# Encode 'id' and 'clinic_type'
label_encoder_id = LabelEncoder()
label_encoder_clinic_type = LabelEncoder()

data['id_encoded'] = label_encoder_id.fit_transform(data['id'])
data['clinic_type_encoded'] = label_encoder_clinic_type.fit_transform(data['clinic_type'])

# Drop original categorical columns to avoid redundancy
data = data.drop(columns=['id', 'clinic_type'])

# Map days of the week to numeric values
day_mapping = {
    'Monday': 0, 'Tuesday': 1, 'Wednesday': 2,
    'Thursday': 3, 'Friday': 4, 'Saturday': 5, 'Sunday': 6
}

data['day_of_week_encoded'] = data['day_of_week'].map(day_mapping)

data = data.dropna()

In [126]:
data

,lastUpdated,queue,wait_time,treat_time,total_time,hour_minute,hour_minute_numeric,day_of_week,hour_time,queue_diff,wait_time_diff,treat_time_diff,total_time_diff,hour_minute_numeric_diff,id_encoded,clinic_type_encoded,day_of_week_encoded
7,2024-10-29 21:17:30.151,4,0.833333,0.550000,1.383333,1900-01-01 21:17:00,1277,Tuesday,21:17:00,0.0,0.000000,0.016667,0.016667,1.0,0,1,1
8,2024-10-29 21:17:30.151,5,1.966667,0.366667,2.333333,1900-01-01 21:17:00,1277,Tuesday,21:17:00,0.0,0.000000,0.016667,0.016667,1.0,3,1,1
9,2024-10-29 21:17:30.151,6,1.216667,0.350000,1.566667,1900-01-01 21:17:00,1277,Tuesday,21:17:00,0.0,0.016667,0.016667,0.033333,1.0,2,1,1
10,2024-10-29 21:17:30.151,4,1.483333,0.483333,1.966667,1900-01-01 21:17:00,1277,Tuesday,21:17:00,-1.0,0.000000,-0.100000,-0.100000,1.0,5,1,1
11,2024-10-29 21:17:30.151,5,1.733333,0.383333,2.116667,1900-01-01 21:17:00,1277,Tuesday,21:17:00,0.0,0.000000,0.016667,0.016667,1.0,6,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
196396,2024-12-01 16:23:30.129,4,0.500000,0.616667,1.116667,1900-01-01 16:23:00,983,Sunday,16:23:00,0.0,0.016667,0.016667,0.033333,1.0,2,1,6
196397,2024-12-01 16:23:30.129,4,1.650000,0.333333,1.983333,1900-01-01 16:23:00,983,Sunday,16:23:00,0.0,0.000000,0.016667,0.016667,1.0,5,1,6
196398,2024-12-01 16:23:30.129,4,1.033333,0.400000,1.433333,1900-01-01 16:23:00,983,Sunday,16:23:00,-1.0,-0.100000,-0.116667,-0.216667,1.0,6,1,6
196399,2024-12-01 16:23:00.000,13,3.200000,2.466667,5.666667,1900-01-01 16:23:00,983,Sunday,16:23:00,0.0,0.000000,0.000000,0.000000,1.0,1,0,6


In [127]:
for col in data.columns:
    print(col)

lastUpdated
queue
wait_time
treat_time
total_time
hour_minute
hour_minute_numeric
day_of_week
hour_time
queue_diff
wait_time_diff
treat_time_diff
total_time_diff
hour_minute_numeric_diff
id_encoded
clinic_type_encoded
day_of_week_encoded


## **Model**

In [128]:
set(data['clinic_type_encoded'])

{0, 1}

In [129]:
import numpy as np

def ts_train_test_split(X, y, group_col, time_col, test_size=0.2, random_state=42):
    """
    Custom train-test split that splits data by group and time, ensuring reproducibility.

    Parameters:
        X (pd.DataFrame): Feature dataset to split.
        y (pd.Series or pd.DataFrame): Target variable to split.
        group_col (str): The column used to group the data.
        time_col (str): The column used to order the data within each group.
        test_size (float): The proportion of data to use for testing (default is 0.2).
        random_state (int): Random seed for reproducibility.

    Returns:
        X_train, X_test, y_train, y_test: Split datasets.
    """
    # Combine X and y for consistent indexing
    data = X.copy()
    data['target'] = y

    np.random.seed(random_state)  # Set random seed for reproducibility
    
    train_indices = []
    test_indices = []

    # Group by the group_col
    for group in data[group_col].unique():
        group_data = data[data[group_col] == group].sort_values(by=time_col)
        n_samples = len(group_data)
        n_test = int(n_samples * test_size)

        # Shuffle indices deterministically
        shuffled_indices = group_data.index.to_numpy()
        np.random.shuffle(shuffled_indices)

        # Split shuffled indices into train and test sets
        train_idx = shuffled_indices[:-n_test]  # All but the last `n_test` samples
        test_idx = shuffled_indices[-n_test:]  # The last `n_test` samples

        train_indices.extend(train_idx)
        test_indices.extend(test_idx)

    # Split into train and test sets
    train_data = data.loc[train_indices]
    test_data = data.loc[test_indices]

    return (
        train_data.drop(columns=['target']),
        test_data.drop(columns=['target']),
        train_data['target'],
        test_data['target']
    )

In [130]:
# Split data into training and testing sets
# WIC == 1, ED == 0

X = data[data['clinic_type_encoded'] == 0][['queue', 'queue_diff', 'hour_minute_numeric', 'hour_minute_numeric_diff', 'id_encoded', 'clinic_type_encoded', 'day_of_week_encoded']]
y = data[data['clinic_type_encoded'] == 0][['wait_time']]

# X = data[data['clinic_type_encoded'] == 1][['queue', 'queue_diff', 'hour_minute_numeric', 'hour_minute_numeric_diff', 'id_encoded', 'clinic_type_encoded', 'day_of_week_encoded']]
# y = data[data['clinic_type_encoded'] == 1][['wait_time']]

# Align indices
X = X.reset_index(drop=True)
y = y.reset_index(drop=True)


X_train, X_test, y_train, y_test = ts_train_test_split(X, y, group_col='clinic_type_encoded', time_col='hour_minute_numeric', test_size=0.1, random_state=42)

In [131]:
def calculate_mape(y_true, y_pred):
    """
    Calculates Mean Absolute Percentage Error (MAPE).
    """
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

def calculate_smape(y_true, y_pred):
    """
    Calculates Symmetric Mean Absolute Percentage Error (SMAPE).
    """
    return 100 * np.mean(np.abs(y_true - y_pred) / ((np.abs(y_true) + np.abs(y_pred)) / 2))

In [132]:
from statsmodels.tsa.arima.model import ARIMA
from sklearn.metrics import mean_squared_error
import itertools
import numpy as np

def evaluate_arima_with_optimization(y_train, y_test, p_range, d_range, q_range):
    """
    Optimizes ARIMA parameters and evaluates the best model.
    """
    best_order = None
    best_mse = float('inf')
    best_rmse = float('inf')
    best_mape = float('inf')
    best_smape = float('inf')
    best_model = None

    # Generate all combinations of p, d, q
    pdq_combinations = list(itertools.product(p_range, d_range, q_range))

    for order in pdq_combinations:
        try:
            # Fit ARIMA model
            model = ARIMA(y_train, order=order)
            model_fit = model.fit()
            
            # Forecast on the test set
            y_pred = model_fit.forecast(steps=len(y_test))
            
            # Calculate Metrics
            mse = mean_squared_error(y_test, y_pred)
            rmse = np.sqrt(mse)
            mape = calculate_mape(y_test, y_pred)
            smape = calculate_smape(y_test, y_pred)

            # Update best model
            if mse < best_mse:
                best_mse, best_rmse, best_mape, best_smape = mse, rmse, mape, smape
                best_order = order
                best_model = model_fit
        except Exception as e:
            print(f"ARIMA fitting failed for order {order}: {e}")
            continue
    
    # Handle cases where no valid model was found
    if best_model is None:
        return None, None, float('nan'), float('nan'), float('nan'), float('nan')
    
    return best_model, best_order, best_mse, best_rmse, best_mape, best_smape

In [133]:
# Define the custom grouped time-series splitter
class GroupedTimeSeriesSplit:
    def __init__(self, group_col, time_col, n_splits):
        self.group_col = group_col
        self.time_col = time_col
        self.n_splits = n_splits

    def split(self, X, y=None):
        """Yield train-test splits for each group."""
        groups = X[self.group_col].unique()
        for group in groups:
            group_data = X[X[self.group_col] == group].sort_values(self.time_col)
            n_samples = len(group_data)
            
            # Skip small groups
            if n_samples <= self.n_splits:
                print(f"Skipping group '{group}' due to insufficient samples.")
                continue
            
            fold_size = n_samples // (self.n_splits + 1)
            if fold_size == 0:
                print(f"Skipping group '{group}' due to insufficient samples per fold.")
                continue

            for i in range(1, self.n_splits + 1):
                start_train = 0
                end_train = fold_size * i
                start_test = fold_size * i
                end_test = fold_size * (i + 1)

                # Ensure indices are within bounds
                if end_train > n_samples or end_test > n_samples:
                    print(f"Skipping fold {i} for group '{group}' due to index out-of-bounds.")
                    continue

                train_idx = group_data.index[start_train:end_train]
                test_idx = group_data.index[start_test:end_test]

                # Ensure indices are not empty
                if len(train_idx) == 0 or len(test_idx) == 0:
                    print(f"Skipping fold {i} for group '{group}' due to empty train/test indices.")
                    continue

                yield train_idx, test_idx

# Define a function for model evaluation
def evaluate_model_with_grouped_cv(model, X, y, splitter, p_range, d_range, q_range):
    mse_scores, rmse_scores, mape_scores, smape_scores, r2_scores = [], [], [], [], []
    
    for train_idx, test_idx in splitter.split(X):
        try:
            # Extract train-test splits
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx].values.ravel(), y.iloc[test_idx].values.ravel()
            
            # Skip if training data is too small
            if len(y_train) < 10:
                print("Skipping split due to insufficient training data.")
                continue
            
            if model == 'ARIMA':
                # Evaluate ARIMA model
                best_model, best_order, mse, rmse, mape, smape = evaluate_arima_with_optimization(
                    y_train, y_test, p_range, d_range, q_range
                )
            else:
                # Evaluate other models
                model.fit(X_train, y_train)
                y_pred = model.predict(X_test)
                
                # # Debug prediction shapes
                # print("y_pred shape:", y_pred.shape)
                # print("y_test shape:", y_test.shape)
                
                mse = mean_squared_error(y_test, y_pred)
                rmse = np.sqrt(mse)
                mape = calculate_mape(y_test, y_pred)
                smape = calculate_smape(y_test, y_pred)
                r2 = r2_score(y_test, y_pred)

            # Append metrics
            mse_scores.append(mse)
            rmse_scores.append(rmse)
            mape_scores.append(mape)
            smape_scores.append(smape)
            r2_scores.append(r2)

        except Exception as e:
            print(f"Error evaluating model: {e}")
            continue
    
    # Return average metrics across folds
    return (
        np.nanmean(mse_scores),
        np.nanmean(rmse_scores),
        np.nanmean(mape_scores),
        np.nanmean(smape_scores),
        np.nanmean(r2_scores),
    )

In [134]:
# Define ARIMA parameter ranges
p_range = range(0, 3)  # AR terms
d_range = range(0, 2)  # Differencing terms
q_range = range(0, 3)  # MA terms

# Define models
models = pd.DataFrame({
    'model_name': ['Linear Regression', 'Random Forest', 'XGBoost', 'ARIMA'],
    'model': [
        LinearRegression(),
        RandomForestRegressor(random_state=42),
        XGBRegressor(random_state=42),
        'ARIMA'  # Placeholder for ARIMA
    ]
})

# Instantiate the custom splitter
custom_splitter = GroupedTimeSeriesSplit(
    group_col='clinic_type_encoded',
    time_col='hour_minute_numeric',
    n_splits=6
)

In [135]:
# Apply the evaluation function
models[['mse', 'rmse', 'mape', 'smape', 'r2']] = models['model'].apply(
    lambda m: pd.Series(evaluate_model_with_grouped_cv(m, X, y, custom_splitter, p_range, d_range, q_range))
)

# Display results
print(models)

Error evaluating model: cannot access local variable 'r2' where it is not associated with a value


/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Error evaluating model: cannot access local variable 'r2' where it is not associated with a value


/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Error evaluating model: cannot access local variable 'r2' where it is not associated with a value


...

/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Error evaluating model: cannot access local variable 'r2' where it is not associated with a value


/Users/derekwu/Desktop/Data/Virtual Environments/my_env/lib/python3.13/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


Error evaluating model: cannot access local variable 'r2' where it is not associated with a value
          model_name                                              model  \
0  Linear Regression                                 LinearRegression()   
1      Random Forest  (DecisionTreeRegressor(max_features=1.0, rando...   
2            XGBoost  XGBRegressor(base_score=None, booster=None, ca...   
3              ARIMA                                              ARIMA   

        mse      rmse       mape      smape        r2  
0  0.448275  0.628853  23.828702  24.210329  0.156356  
1  0.233884  0.476413  18.019005  17.263824  0.493691  
2  0.235995  0.476942  17.883349  17.127677  0.488122  
3  0.582305  0.741601  28.069547  26.418688       NaN  


/var/folders/my/1zd7hfx53qq246r8xfrrzkdc0000gn/T/ipykernel_64418/1780320840.py:98: RuntimeWarning: Mean of empty slice
  np.nanmean(r2_scores),
